In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

import warnings
warnings.simplefilter('ignore')

## Исполнительные производства 

In [3]:
API_KEY = 'cPPfK9ZPMYIUMydx'
inn_ = '7428007317'

url = f'https://api.checko.ru/v2/enforcements?key={API_KEY}&inn={inn_}'

In [9]:
url = f'https://api.checko.ru/v2/legal-cases?key={API_KEY}&inn={inn_}&role=defendant&actual=true'

In [10]:
response = requests.get(url)

In [12]:
enforcements = pd.DataFrame(response.json()['data']['Записи'])

## Название организации

In [74]:
def get_company_info_by_ogrn(ogrn):
    url = f"https://kontragent.vbr.ru/company/?ogrn={ogrn}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Проверим, что запрос успешен

        soup = BeautifulSoup(response.text, 'html.parser')

        # --- Способ 1: Парсим основной блок с информацией (самый надежный) ---
        # Ищем все блоки с классом CardRow_card-row__PqkUn
        card_rows = soup.find_all('div', class_=lambda c: c and 'CardRow_card-row__PqkUn' in c)

        full_name = "Не найдено"
        short_name = "Не найдено"
        inn = "Не найдено"
        ogrn_parsed = "Не найдено"

        for row in card_rows:
            title_tag = row.find('p', class_=lambda c: c and 'CardRow_title__IgmhN' in c)
            if not title_tag:
                continue

            title_text = title_tag.get_text(strip=True)
            value_tag = row.find('p', class_=lambda c: c and 'CardRow_text__lkZ_w' in c)
            if not value_tag:
                continue

            value_text = value_tag.get_text(strip=True)

            if "Полное наименование" in title_text:
                full_name = value_text
            elif "Краткое наименование" in title_text:
                short_name = value_text
            elif "ИНН" in title_text:
                inn = value_text
            elif "ОГРН" in title_text:
                ogrn_parsed = value_text

        # --- Способ 2: Простой парсинг из <title> на случай, если первый способ не сработает ---
        if full_name == "Не найдено" or short_name == "Не найдено":
            title_tag = soup.find('title')
            if title_tag:
                title_text = title_tag.get_text(strip=True)
                # Из "АО \"А/К 1114\": ИНН 3903008670..." можно вытащить краткое имя
                if ':' in title_text:
                    short_name_from_title = title_text.split(':')[0].strip()
                    # Это может быть и кратким названием, но для полного лучше использовать og:description
                    if short_name == "Не найдено":
                        short_name = short_name_from_title

            # Ищем полное название в og:description
            og_desc_meta = soup.find('meta', property='og:description')
            if og_desc_meta and og_desc_meta.get('content'):
                # В og:description часто есть полное название в начале
                desc_text = og_desc_meta['content']
                if full_name == "Не найдено":
                    # Простой эвристический поиск: берем текст до "в России" или до "✅"
                    import re
                    match = re.search(r'^(.*?)(?:\s+в России|✅)', desc_text)
                    if match:
                        full_name = match.group(1).strip()
                    else:
                        full_name = desc_text.split('в России')[0].strip() if 'в России' in desc_text else desc_text

        return {
            "full_name": full_name,
            "short_name": short_name,
            "inn": inn,
            "ogrn": ogrn_parsed
        }

    except requests.exceptions.RequestException as e:
        print(f"Ошибка при загрузке страницы: {e}")
        return None
    except Exception as e:
        print(f"Произошла ошибка при парсинге: {e}")
        return None

In [76]:
get_company_info_by_ogrn('1027700068440')

Ошибка при загрузке страницы: 404 Client Error: Not Found for url: https://kontragent.vbr.ru/company/?ogrn=1027700068440


## Размер предприятия

In [14]:
data = pd.DataFrame([
    ['Крупные предприятия', 2000000],
    ['Средние предприятия', 800000],
    ['Малые предприятия', 120000],
    ['Микропредприятия', 0]
], columns=['business_segment', 'revenue'])\
.sort_values(by='revenue', ascending=True)

In [15]:
data.to_csv('input/b_segment_mapping.csv', index=False)

## Коды субъектов РФ

In [37]:
def parse_region_codes(url):
    """
    Парсит справочник кодов регионов с сайта consultant.ru.
    Внимание: Документ утратил силу, но код демонстрирует парсинг.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'
    except requests.RequestException as e:
        print(f"Ошибка при загрузке страницы: {e}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Ищем таблицу с кодом региона (обычно это <table> внутри <div class="document-page">)
    # Вариант 1: Ищем таблицу, содержащую "Код" и "Наименование"
    table = None
    for tbl in soup.find_all('table'):
        if tbl.find(string='Код') and tbl.find(string='Наименование'):
            table = tbl
            break
    
    if not table:
        # Вариант 2: Ищем <pre> блок или <div> с данными, если таблица не найдена
        pre_blocks = soup.find_all('pre')
        for pre in pre_blocks:
            if 'Республика Адыгея' in pre.text or '01' in pre.text:
                return parse_from_pre(pre.text)
        
        # Вариант 3: Если данные в текстовом виде без таблицы
        content_div = soup.find('div', class_='document-page')
        if content_div:
            text = content_div.get_text()
            if 'Код' in text and 'Наименование' in text:
                return parse_from_text(text)
        
        print("Таблица не найдена. Структура страницы могла измениться.")
        return None

    # Парсим таблицу
    data = []
    rows = table.find_all('tr')
    
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 2:
            code = cols[0].get_text(strip=True)
            name = cols[1].get_text(strip=True)
            
            # Пропускаем заголовки
            if code == 'Код' or code == '' or not code.isdigit():
                continue
            
            data.append({
                'region_code': code,
                'region': name
            })
    
    return data

def parse_from_pre(text):
    """Парсит данные из <pre> блока, если таблица не найдена"""
    lines = text.strip().split('\n')
    data = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Пытаемся разделить код и название
        parts = line.split(maxsplit=1)
        if len(parts) == 2 and parts[0].isdigit():
            data.append({
                'region_code': parts[0],
                'region': parts[1].strip()
            })
    
    return data if data else None

def parse_from_text(text):
    """Парсит данные из обычного текста, если таблица не найдена"""
    import re
    
    # Ищем паттерн: код (2-3 цифры) + название региона
    pattern = r'(\d{2,3})\s+(.+?)(?=\n\d{2,3}\s+|\Z)'
    matches = re.findall(pattern, text, re.DOTALL)
    
    data = []
    for code, name in matches:
        name = name.strip().replace('\n', ' ').replace('  ', ' ')
        data.append({
            'region_code': code,
            'region': name
        })
    
    return data if data else None

def save_to_csv(data, filename):
    """Сохраняет данные в CSV файл"""
    if not data:
        return
    
    df = pd.DataFrame(data)
    df.to_csv(f'{filename}.csv', index=False, encoding='utf-8-sig')
    print(f"Данные сохранены в {filename}")

In [40]:
# url = "https://www.consultant.ru/document/cons_doc_LAW_108669/88a12659e7cc781c56303430d98ae6c8a683892a/"
url = 'https://www.consultant.ru/document/cons_doc_LAW_530183/cce626b2bbad437352e50887f1e72e90e98a79d4/'
region_data = parse_region_codes(url)

if region_data:
    save_to_csv(region_data, 'input/region_mapping')
else:
    print("Не удалось спарсить данные. Проверьте структуру страницы.")

Данные сохранены в input/region_mapping


## ОКТМО

In [16]:
url_data = 'https://rosstat.gov.ru/opendata/7708234640-oktmo/data-20260601T1406-structure-20260210T1102.csv'
url_columns = 'https://rosstat.gov.ru/opendata/7708234640-oktmo/structure-20260210T1102.csv'

try:
    response_data = requests.get(url_data, verify=False)
    response_data.raise_for_status()
    
    response_columns = requests.get(url_columns, verify=False)
    response_columns.raise_for_status()
    
    meta = pd.read_csv(StringIO(response_columns.text), delimiter=';', encoding='utf-8')
    data = pd.read_csv(StringIO(response_data.text), delimiter=';', encoding='utf-8', header=None, dtype=str)
    data.columns = meta['field name']

except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке данных: {e}")

C:\Users\123\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\123\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [44]:
oktmo_mapping = pd.DataFrame()
oktmo_mapping['oktmo'] = data['TER'] + data['KOD1'] + data['KOD2'] + data['KOD3']
oktmo_mapping['oktmo_explain'] = data['NAME1']

In [46]:
oktmo_mapping.to_csv('input/oktmo_mapping.csv', index=False, encoding='utf-8-sig')

## ОКОПФ

In [52]:
url_data = 'https://rosstat.gov.ru/opendata/7708234640-okopf/data-20251001T1510-structure-20180326T1603.csv'
url_columns = 'https://rosstat.gov.ru/opendata/7708234640-okopf/structure-20180326T1603.csv'

try:
    response_data = requests.get(url_data, verify=False)
    response_data.raise_for_status()
    
    response_columns = requests.get(url_columns, verify=False)
    response_columns.raise_for_status()
    
    meta = pd.read_csv(StringIO(response_columns.text), delimiter=',', encoding='utf-8')
    data = pd.read_csv(StringIO(response_data.text), delimiter=';', encoding='utf-8', header=None, dtype=str)
    data.columns = meta['field name']

except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке данных: {e}")

C:\Users\123\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\123\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [60]:
okopf_mapping = pd.DataFrame()
okopf_mapping['okopf'] = data['Code'].str.replace(' ', '')
okopf_mapping['okopf_explain'] = data['Name']

In [65]:
okopf_mapping.to_csv('input/okopf_mapping.csv', index=False, encoding='utf-8-sig')

## ОКОГУ 

In [31]:
url_data = 'https://rosstat.gov.ru/opendata/7708234640-okogu/data-20260401T1704-structure-20231215T1512.csv'
url_columns = 'https://rosstat.gov.ru/opendata/7708234640-okogu/structure-20231215T1512.csv'

try:
    response_data = requests.get(url_data, verify=False)
    response_data.raise_for_status()
    
    response_columns = requests.get(url_columns, verify=False)
    response_columns.raise_for_status()
    
    meta = pd.read_csv(StringIO(response_columns.text), delimiter=';', encoding='utf-8')
    data = pd.read_csv(StringIO(response_data.text), delimiter=';', encoding='utf-8', header=None, dtype=str, usecols=[0,1])
    data.columns = meta['field name']

except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке данных: {e}")

In [33]:
okogu_mapping = pd.DataFrame()
okogu_mapping['okogu'] = data['code']
okogu_mapping['okogu_explain'] = data['name']

In [34]:
okogu_mapping.to_csv('input/okogu_mapping.csv', index=False, encoding='utf-8-sig')

## ОКФС 

In [37]:
url_data = 'https://rosstat.gov.ru/opendata/7708234640-okfs/data-20231101T1111-structure-20180326T1603.csv'
url_columns = 'https://rosstat.gov.ru/opendata/7708234640-okfs/structure-20180326T1603.csv'

try:
    response_data = requests.get(url_data, verify=False)
    response_data.raise_for_status()
    
    response_columns = requests.get(url_columns, verify=False)
    response_columns.raise_for_status()
    
    meta = pd.read_csv(StringIO(response_columns.text), delimiter=',', encoding='utf-8')
    data = pd.read_csv(StringIO(response_data.text), delimiter=';', encoding='utf-8', header=None, dtype=str, usecols=[0,1])
    data.columns = meta['field name']

except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке данных: {e}")

In [40]:
okfc_mapping = pd.DataFrame()
okfc_mapping['okfc'] = data['Code']
okfc_mapping['okfc_explain'] = data['Name']

In [42]:
okfc_mapping.to_csv('input/okfc_mapping.csv', index=False, encoding='utf-8-sig')

## ОКВЭД

In [2]:
url_data = 'https://rosstat.gov.ru/opendata/7708234640-okvedva/data-20260601T1406-structure-20180402T1704.csv'
url_columns = 'https://rosstat.gov.ru/opendata/7708234640-okvedva/structure-20180402T1704.csv'

try:
    response_data = requests.get(url_data, verify=False)
    response_data.raise_for_status()
    
    response_columns = requests.get(url_columns, verify=False)
    response_columns.raise_for_status()
    
    meta = pd.read_csv(StringIO(response_columns.text), delimiter=',', encoding='utf-8')
    data = pd.read_csv(StringIO(response_data.text), delimiter=';', encoding='utf-8', header=None, dtype=str)
    data.columns = meta['field name']

except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке данных: {e}")

In [46]:
okved_mapping = pd.DataFrame()
okved_mapping['okved'] = data['Code']
okved_mapping['okved_explain'] = data['Name']

In [37]:
def fix_okved(code):
    parts = str(code).split('.')
    if (len(parts) == 3) and (len(parts[2]) == 1):
        parts[2] = parts[2] + '0'
        return '.'.join(parts)
    else:
        return code

In [47]:
okved_mapping['okved'] = okved_mapping['okved'].apply(fix_okved)

In [48]:
okved_mapping.to_csv('input/okved_mapping.csv', index=False, encoding='utf-8-sig')